# NB15 — CH4 Hardware Mod: 820 kΩ → 410 kΩ Bias Resistor

**Version:** 1 · **Author:** CB · **Date:** 2026-07-01  
**Offline notebook** — runs from saved data in `data/ch4_resistor_test/20260701_151310/`.
Acquisition cells (§ 3) are marked *hardware required* and can be skipped.

---

## 1  Background

On 2026-07-01 a second 820 kΩ resistor was soldered **in parallel** on CH4,
reducing the bias/discharge resistance: $R_\text{eff} = 820\,\text{k}\Omega \| 820\,\text{k}\Omega = 410\,\text{k}\Omega$.

The piezo sensor forms a high-pass RC filter with its own capacitance $C_\text{piezo}$:
$$f_c = \frac{1}{2\pi R C_\text{piezo}}$$
Halving $R$ **doubles $f_c$**, shifting the low-frequency roll-off upward while
also reducing the Johnson-noise contribution ($V_n \propto \sqrt{R}$).

**Expected effects**
- Slight loss of sensitivity below $f_c$ (higher corner frequency)
- Improved noise floor / SNR above $f_c$
- Lower broadband RMS noise

| Channel | Bias R (before) | Bias R (after) | Role |
|---------|-----------------|----------------|------|
| CH1 | 820 kΩ | 820 kΩ | Soil piezo — **control** |
| CH2 | — | — | Defective, disabled |
| CH3 | 820 kΩ | 820 kΩ | Rod piezo A — **control** |
| CH4 | 820 kΩ | **410 kΩ** | Rod piezo B — **modified** |

## 2  Method

Reference-normalised differential to isolate the resistor effect from
environmental and temporal drift:

$$\Delta_\text{net}(f) =
    \underbrace{\bigl[\mathrm{PSD}_{\text{CH4,after}} - \mathrm{PSD}_{\text{CH4,before}}\bigr]}_{\text{CH4 change}}
    - \underbrace{\frac{1}{2}\bigl[\Delta_{\text{CH1}} + \Delta_{\text{CH3}}\bigr]}_{\text{reference drift}}$$

- **Before** baseline: 17 hourly PSD snapshots from session `20260630_025848`
  (all channels at 820 kΩ, overnight, 03:00–18:00 UTC)
- **After** data: 30 passive frames acquired 2026-07-01 ~15:13–17:47 UTC
  (session `20260701_151310`)

---

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as spsig
from scipy.interpolate import interp1d
from scipy.ndimage import uniform_filter1d
%matplotlib inline

from plant_ae.watering import CHARACTERIZATION_ROOT
CHAR_ROOT = CHARACTERIZATION_ROOT
if str(CHAR_ROOT) not in sys.path:
    sys.path.insert(0, str(CHAR_ROOT))
if str(CHAR_ROOT / 'instrument_control') not in sys.path:
    sys.path.insert(0, str(CHAR_ROOT / 'instrument_control'))

from plant_ae.rolling import BAND_EDGES_HZ

# ── Session IDs ───────────────────────────────────────────────────────────────
BEFORE_SESSION = '20260630_025848'    # 820 kΩ on all channels
AFTER_RUN_ID   = '20260701_151310'    # CH4 modified to 410 kΩ

SNAPSHOT_DIR = (CHAR_ROOT / 'data' / 'reports' / 'notebooks'
                / '04_continuous_frequency_sweep' / BEFORE_SESSION / 'psd_snapshots')
AFTER_DIR    = CHAR_ROOT / 'data' / 'ch4_resistor_test' / AFTER_RUN_ID

CHANNELS  = (1, 3, 4)
IDX       = {1: 0, 3: 1, 4: 2}
CH_LABELS = {1: 'CH1  820 kΩ (control)', 3: 'CH3  820 kΩ (control)', 4: 'CH4  410 kΩ (modified)'}
CH_COLORS = {1: '#1f77b4', 3: '#2ca02c', 4: '#d62728'}
NPERSEG   = 16384

print(f'Before session : {BEFORE_SESSION}')
print(f'After run      : {AFTER_RUN_ID}')
print(f'After data     : {AFTER_DIR}')

## 3  Load Data

In [ ]:
# ── Before: average hourly PSD snapshots ──────────────────────────────────────
snap_files = sorted(SNAPSHOT_DIR.glob('psd_snapshot_*.npz'))
print(f'Before snapshots : {len(snap_files)}')

all_psds = []
freq_before = None
for f in snap_files:
    d = np.load(f)
    all_psds.append(d['mean_psd'])
    if freq_before is None:
        freq_before = d['frequencies'].astype(np.float64)

psd_before = np.mean(np.stack(all_psds), axis=0)   # (3, N_freq)

# ── After: load saved voltages ────────────────────────────────────────────────
d_after   = np.load(AFTER_DIR / 'voltages_after.npz')
fs_hz     = float(d_after['sample_rate_hz'])
voltages  = {1: d_after['ch1'], 3: d_after['ch3'], 4: d_after['ch4']}
N_FRAMES  = voltages[1].shape[0]

print(f'After frames     : {N_FRAMES}  (fs={fs_hz/1e3:.0f} kSa/s)')
print()

# Sanity: broadband RMS
for ch in CHANNELS:
    rms = float(np.sqrt(np.mean(voltages[ch]**2))) * 1e3
    print(f'  {CH_LABELS[ch]}:  RMS = {rms:.2f} mV')

## 4  Welch PSD and Differential Analysis

In [ ]:
def welch_mean(V, fs, nperseg):
    psds = [spsig.welch(v, fs=fs, nperseg=nperseg, window='hann', scaling='density')[1]
            for v in V]
    return spsig.welch(V[0], fs=fs, nperseg=nperseg, window='hann', scaling='density')[0], \
           np.mean(np.stack(psds), axis=0)

psd_after_w = {}
freq_w = None
for ch in CHANNELS:
    fw, pw = welch_mean(voltages[ch], fs_hz, NPERSEG)
    psd_after_w[ch] = pw
    freq_w = fw

# Interpolate before-PSD onto Welch grid
psd_before_w = {}
for ch in CHANNELS:
    f_interp = interp1d(freq_before, psd_before[IDX[ch]], kind='linear',
                        bounds_error=False, fill_value='extrapolate')
    psd_before_w[ch] = np.maximum(f_interp(freq_w), 1e-30)

def db(p):
    return 10 * np.log10(np.maximum(p, 1e-30))

delta     = {ch: db(psd_after_w[ch]) - db(psd_before_w[ch]) for ch in CHANNELS}
ref_avg   = 0.5 * (delta[1] + delta[3])
delta_eff = delta[4] - ref_avg
SMOOTH    = 15

mask = (freq_w >= 500) & (freq_w <= 200_000)
fk   = freq_w[mask] / 1e3

# ── Figure ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
fig.suptitle(
    f'CH4 820 kΩ → 410 kΩ  |  Before: {BEFORE_SESSION}  |  After: {AFTER_RUN_ID}\n'
    f'N={N_FRAMES} frames after · Welch nperseg={NPERSEG} · 17 hourly snapshots before',
    fontsize=10)

ax = axes[0]
ax.set_title('Averaged Welch PSD — After (solid) vs Before (dashed)', fontsize=9)
for ch in CHANNELS:
    ax.plot(fk, db(psd_after_w[ch])[mask],  lw=0.9, color=CH_COLORS[ch], label=CH_LABELS[ch])
    ax.plot(fk, db(psd_before_w[ch])[mask], lw=0.6, color=CH_COLORS[ch], ls='--', alpha=0.5)
ax.set_ylabel('PSD (dB re V²/Hz)')
ax.legend(fontsize=8); ax.grid(True, lw=0.3, alpha=0.5)

ax = axes[1]
ax.set_title('Raw ΔPSD per channel  (After − Before)', fontsize=9)
for ch in CHANNELS:
    ax.plot(fk, uniform_filter1d(delta[ch], SMOOTH)[mask],
            lw=0.9, color=CH_COLORS[ch], label=CH_LABELS[ch])
ax.axhline(0, color='k', lw=0.5, ls=':')
ax.set_ylabel('ΔdB'); ax.legend(fontsize=8); ax.grid(True, lw=0.3, alpha=0.5)

ax = axes[2]
ax.set_title('Reference-normalised Δnet  (CH4 change − mean(CH1, CH3) change)', fontsize=9)
sm = uniform_filter1d(delta_eff, SMOOTH)
ax.plot(fk, sm[mask], lw=1.0, color=CH_COLORS[4], label='Δnet CH4 (smoothed)')
ax.fill_between(fk, sm[mask], 0, where=(sm[mask] > 0), alpha=0.15, color=CH_COLORS[4])
ax.fill_between(fk, sm[mask], 0, where=(sm[mask] < 0), alpha=0.15, color='gray')
ax.axhline(0, color='k', lw=0.5, ls=':')
ax.set_ylabel('ΔdB (net effect)'); ax.set_xlabel('Frequency (kHz)')
ax.legend(fontsize=8); ax.grid(True, lw=0.3, alpha=0.5)

plt.tight_layout()
fig.savefig(AFTER_DIR / 'psd_differential.png', dpi=150, bbox_inches='tight')
plt.show()

## 5  Band-Level Summary Table

In [ ]:
bands   = list(zip(BAND_EDGES_HZ[:-1], BAND_EDGES_HZ[1:]))
blabels = [f'{int(lo/1e3)}-{int(hi/1e3)} kHz' for lo, hi in bands]

def band_db(psd_dict, ch, lo, hi):
    m = (freq_w >= lo) & (freq_w < hi)
    return 10 * np.log10(max(float(np.trapezoid(psd_dict[ch][m], freq_w[m])), 1e-30))

rows = []
for (lo, hi), lbl in zip(bands, blabels):
    b   = {ch: band_db(psd_before_w, ch, lo, hi) for ch in CHANNELS}
    a   = {ch: band_db(psd_after_w,  ch, lo, hi) for ch in CHANNELS}
    ref = 0.5 * ((a[1]-b[1]) + (a[3]-b[3]))
    net = (a[4]-b[4]) - ref
    rows.append((lbl, b[1], a[1], b[3], a[3], b[4], a[4], net))

print(f'{"Band":<14}  {"Bef CH1":>8}  {"Aft CH1":>8}  {"Bef CH3":>8}  {"Aft CH3":>8}  '
      f'{"Bef CH4":>8}  {"Aft CH4":>8}  {"Δnet":>7}')
print('-' * 82)
for r in rows:
    lbl, b1, a1, b3, a3, b4, a4, net = r
    marker = ' ◀' if abs(net) >= 2.0 else ''
    print(f'{lbl:<14}  {b1:>8.1f}  {a1:>8.1f}  {b3:>8.1f}  {a3:>8.1f}  '
          f'{b4:>8.1f}  {a4:>8.1f}  {net:>+7.1f}{marker}')

print()
nets = [r[-1] for r in rows]
nets_5_100 = [r[-1] for r in rows if '-' in r[0] and int(r[0].split('-')[0]) >= 5
              and int(r[0].split('-')[1].replace(' kHz','')) <= 100]
print(f'Median Δnet (all bands)   : {np.median(nets):+.1f} dB')
print(f'Median Δnet (5–100 kHz)   : {np.median(nets_5_100):+.1f} dB')
print(f'Max Δnet                  : {max(nets):+.1f} dB')
print(f'Min Δnet                  : {min(nets):+.1f} dB')

## 6  Results and Decision

### Findings (2026-07-01, session `20260701_151310`)

| Frequency range | Δnet CH4 (ref-normalised) | Interpretation |
|---|---|---|
| 0–5 kHz | **−2.2 dB** | Expected: higher $f_c$ attenuates low end |
| 5–100 kHz | **+1 to +3 dB** | Improved sensitivity across AE band |
| 25–30 kHz | **+2.8 dB** (peak) | Rod structural resonance band |
| 60–65 kHz | **+2.6 dB** (peak) | Ultrasonic AE band |
| 100–250 kHz | +0 to +2 dB | Diminishing gain near Nyquist |

**Broadband RMS:** CH4 = 80.9 mV vs CH3 = 82.7 mV (−1.8 mV, i.e. lower noise floor from halved Johnson noise)

### Decision: **Keep 410 kΩ on CH4**

The modification consistently improves CH4 sensitivity in the 5–100 kHz AE band
by 1–3 dB (relative to unchanged CH1/CH3), with a slight reduction below ~5 kHz.
Since plant AE cavitation events are predominantly above 20 kHz, the trade-off
is favourable. The resistor change is permanent.

### Plant status

Plant was heat-stressed (~1 week prior, too much direct sun) and in shadow
recovery at the time of the test. Stressed plants show elevated xylem tension
and are expected to produce more cavitation AE — the ongoing Longrun session
`20260701_174644` monitors this.

---

### Hardware log

| Date | Channel | Change | Config |
|------|---------|--------|--------|
| 2026-06-21 | CH4 | Initial setup | 820 kΩ + LM358 + piezo + metal rod |
| **2026-07-01** | **CH4** | **+820 kΩ parallel** | **820 kΩ ‖ 820 kΩ = 410 kΩ** |

---

## Appendix A: Acquisition (hardware required — already done)

The cell below was used to acquire the 30 frames stored in `20260701_151310/voltages_after.npz`.
It is kept for reproducibility. **Skip when running offline.**

In [ ]:
# ── SKIP — data already saved ─────────────────────────────────────────────────
# Uncomment to re-acquire (overwrites voltages_after.npz in a new timestamped dir).

# from scope.config import load_config
# from scope.instrument import InstrumentConnection
# from plant_ae.deep_acquisition import (
#     configure_deep_memory_scope, acquire_deep_memory_frame,
#     DEFAULT_SAMPLE_RATE_HZ, DEFAULT_MEMORY_DEPTH, ExperimentConfig,
# )
# from plant_ae.watering import CONFIG_PATH
# from datetime import datetime, timezone
#
# NEW_RUN_ID  = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
# NEW_OUT_DIR = CHAR_ROOT / 'data' / 'ch4_resistor_test' / NEW_RUN_ID
# NEW_OUT_DIR.mkdir(parents=True, exist_ok=True)
#
# exp_config = ExperimentConfig()
# config     = load_config(CONFIG_PATH)
# voltages_acq = {ch: [] for ch in CHANNELS}
# fs_acq = None
# i = 0
# while i < 30:
#     try:
#         with InstrumentConnection(config) as conn:
#             info = configure_deep_memory_scope(conn)
#             fs_acq = info['sample_rate_hz']
#             while i < 30:
#                 caps = acquire_deep_memory_frame(conn, exp_config, sequence=i, channels=CHANNELS)
#                 for cap in caps:
#                     voltages_acq[int(cap.metadata.channel)].append(
#                         np.asarray(cap.voltage_vector, dtype=np.float32))
#                 i += 1
#                 print(f'  [{i}/30]')
#     except Exception as e:
#         print(f'  Reconnect: {e}'); __import__('time').sleep(5)
# for ch in CHANNELS:
#     voltages_acq[ch] = np.stack(voltages_acq[ch])
# np.savez_compressed(NEW_OUT_DIR / 'voltages_after.npz',
#                     ch1=voltages_acq[1], ch3=voltages_acq[3], ch4=voltages_acq[4],
#                     sample_rate_hz=np.array(fs_acq))
# print(f'Saved → {NEW_OUT_DIR}')
print('Acquisition cell skipped (data already in', AFTER_DIR, ')')